# Global Detector — LSH Variant

Same as `detector_global_ts.ipynb`, but includes the Locality-Sensitive Hashing (LSH) geometric distillation step first: standardizes the data, hashes it into similarity buckets via random hyperplanes, and keeps one representative sample per bucket before training the detector. Threshold reported in the thesis for this variant: 0.40.

In [ ]:
import os, pyreadr, joblib, gc, optuna, warnings, time
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ==============================================================================
# NOTE: paths below are relative to this notebook living in /notebooks.
# Place the TEP dataset files in /data and trained model artifacts in /models
# (see the repository README for download links and folder layout).
# 0. CONFIGURAÇÕES GERAIS
# ==============================================================================
PASTA_MODELOS = os.path.join("..", "models", "lsh")
CAMINHO_NORMAL = os.path.join("..", "data", "TEP_FaultFree_Training.RData")
CAMINHO_FALHAS = os.path.join("..", "data", "TEP_Faulty_Training.RData")
FALHAS_EXCLUIDAS = [3, 9, 15]
USAR_CUDA = True  # Mude para False se não tiver GPU NVIDIA

if not os.path.exists(PASTA_MODELOS):
    os.makedirs(PASTA_MODELOS)

# FUNÇÃO LSH
def destilar_via_lsh(df, scaler, hiperplanos):
    if len(df) == 0:
        return df
    cols_ignorar = ['faultNumber', 'simulationRun', 'sample', 'target', 'group_id']
    features = df.drop(columns=[c for c in cols_ignorar if c in df.columns], errors='ignore')
    features = features[scaler.feature_names_in_]
    feat_scaled = scaler.transform(features)
    projecoes = np.dot(feat_scaled, hiperplanos)
    hashes = (projecoes > 0).astype(int)
    df_temp = df.copy()
    df_temp['lsh_bucket'] = [''.join(row.astype(str)) for row in hashes]
    df_distil = df_temp.sample(frac=1, random_state=42).drop_duplicates(subset=['lsh_bucket'])
    return df_distil.drop(columns=['lsh_bucket'], errors='ignore')

# FUNÇÃO TOPSIS
def aplicar_topsis(df, colunas_beneficio, colunas_custo, pesos=None):
    """
    Aplica TOPSIS em um DataFrame.
    - colunas_beneficio: critérios a maximizar
    - colunas_custo: critérios a minimizar
    - pesos: lista/array de pesos para todos os critérios, na ordem:
             colunas_beneficio + colunas_custo
    """
    criterios = colunas_beneficio + colunas_custo
    matriz = df[criterios].astype(float).values

    # Normalização vetorial
    norma = np.sqrt((matriz ** 2).sum(axis=0))
    norma[norma == 0] = 1.0
    matriz_norm = matriz / norma

    # Pesos
    if pesos is None:
        pesos = np.ones(len(criterios)) / len(criterios)
    else:
        pesos = np.array(pesos, dtype=float)
        pesos = pesos / pesos.sum()

    matriz_peso = matriz_norm * pesos

    # Solução ideal positiva e negativa
    ideal_positivo = np.zeros(len(criterios))
    ideal_negativo = np.zeros(len(criterios))

    for i, criterio in enumerate(criterios):
        if criterio in colunas_beneficio:
            ideal_positivo[i] = np.max(matriz_peso[:, i])
            ideal_negativo[i] = np.min(matriz_peso[:, i])
        else:  # custo
            ideal_positivo[i] = np.min(matriz_peso[:, i])
            ideal_negativo[i] = np.max(matriz_peso[:, i])

    # Distâncias
    dist_pos = np.sqrt(((matriz_peso - ideal_positivo) ** 2).sum(axis=1))
    dist_neg = np.sqrt(((matriz_peso - ideal_negativo) ** 2).sum(axis=1))

    # Coeficiente de proximidade
    denom = dist_pos + dist_neg
    denom[denom == 0] = 1.0
    score_topsis = dist_neg / denom

    return score_topsis

# ==============================================================================
# 1. CARREGAMENTO E PREPARAÇÃO (RIGOR DE ENGENHARIA)
# ==============================================================================
print("🚀 Iniciando treinamento do DETECTOR GLOBAL (Versão .RData)...")
tempo_inicio = time.time()

# 1.1 Carregar Base Normal
print("⏳ Lendo base Normal (.RData)...")
res_n = pyreadr.read_r(CAMINHO_NORMAL)
df_normal = res_n[list(res_n.keys())[0]]
df_normal['target'] = 0
# Criar ID único para simulações normais
df_normal['group_id'] = "norm_" + df_normal['simulationRun'].astype(str)
del res_n

# 1.2 Carregar Base de Falhas
print("⏳ Lendo base de Falhas (.RData)...")
res_f = pyreadr.read_r(CAMINHO_FALHAS)
df_falhas = res_f[list(res_f.keys())[0]]
# Filtro Crítico (3, 9, 15)
df_falhas = df_falhas[~df_falhas['faultNumber'].isin(FALHAS_EXCLUIDAS)]
df_falhas['target'] = 1
# Criar ID único para simulações de falha
df_falhas['group_id'] = "fault_" + df_falhas['faultNumber'].astype(str) + "_" + df_falhas['simulationRun'].astype(str)
del res_f
gc.collect()

# ==============================================================================
# 1.3 SELEÇÃO POR TRAJETÓRIA (RIGOR DE ENGENHARIA - RUNS INTEIROS)
# ==============================================================================
# Cada Run tem 500 amostras. 200 Runs = 100.000 amostras.
N_RUNS_ALVO = 200

print(f"📉 Selecionando {N_RUNS_ALVO} simulações inteiras de cada classe...")

# 1. Identificar IDs únicos de cada classe
runs_normais = df_normal['group_id'].unique()
runs_falhas = df_falhas['group_id'].unique()

# 2. Selecionar os Runs aleatoriamente (Garante trajetórias completas)
rng = np.random.RandomState(42)
runs_n_esc = rng.choice(runs_normais, size=min(N_RUNS_ALVO, len(runs_normais)), replace=False)
runs_f_esc = rng.choice(runs_falhas, size=min(N_RUNS_ALVO, len(runs_falhas)), replace=False)

# 3. Filtrar os DataFrames originais pelos Runs escolhidos
df_normal_sample = df_normal[df_normal['group_id'].isin(runs_n_esc)]
df_falhas_sample = df_falhas[df_falhas['group_id'].isin(runs_f_esc)]

# 4. Montagem do Dataset Final
df_completo = pd.concat([df_normal_sample, df_falhas_sample], ignore_index=True)

# 5. LIMPEZA DE MEMÓRIA IMEDIATA (Essencial para não travar o PC)
print("🧹 Libertando memória RAM...")
del df_normal, df_falhas, df_normal_sample, df_falhas_sample
gc.collect()

# ==============================================================================
# 2. DIVISÃO, GEOMETRIA LSH E BALANCEAMENTO
# ==============================================================================
grupos_unicos = df_completo['group_id'].unique()
np.random.RandomState(42).shuffle(grupos_unicos)
corte = int(len(grupos_unicos) * 0.7)
grupos_treino = grupos_unicos[:corte]

df_train_bruto = df_completo[df_completo['group_id'].isin(grupos_treino)].copy()
df_test = df_completo[~df_completo['group_id'].isin(grupos_treino)].copy()

colunas_drop = ['faultNumber', 'simulationRun', 'sample', 'target', 'group_id']
features = [c for c in df_completo.columns if c not in colunas_drop]

# --- 2.1 CRIANDO A GEOMETRIA LSH GLOBAL ---
print("📐 Gerando Geometria Matemática Global (Scaler + Hiperplanos)...")
# O Scaler aprende APENAS com o treino (para evitar data leakage)
scaler_global = StandardScaler().fit(df_train_bruto[features])
num_ands = 16
hiperplanos_globais = np.random.RandomState(42).randn(len(features), num_ands)

# Salva a geometria para os especialistas usarem depois!
joblib.dump((scaler_global, hiperplanos_globais), os.path.join(PASTA_MODELOS, "geometria_lsh.joblib"))

# --- 2.2 DESTILAÇÃO LSH DO TREINO ---
print("🧩 Destilando a base de TREINO via LSH...")
df_train_norm = df_train_bruto[df_train_bruto['target'] == 0]
df_train_falt = df_train_bruto[df_train_bruto['target'] == 1]

df_norm_clean = destilar_via_lsh(df_train_norm, scaler_global, hiperplanos_globais)
df_falt_clean = destilar_via_lsh(df_train_falt, scaler_global, hiperplanos_globais)

# --- 2.3 BALANCEAMENTO 50/50 APÓS LSH ---
n_tr = min(len(df_norm_clean), len(df_falt_clean))
df_train_final = pd.concat([
    df_norm_clean.sample(n=n_tr, random_state=42),
    df_falt_clean.sample(n=n_tr, random_state=42)
]).reset_index(drop=True)

# Define as variáveis que o Optuna e o Modelo vão usar
X_train, y_train, groups_train = df_train_final[features], df_train_final['target'], df_train_final['group_id']
X_test, y_test = df_test[features], df_test['target']

print(f"✅ LSH e Split concluídos!")
print(f"   Treino (Destilado/Balanceado): {len(X_train)} amostras")
print(f"   Teste (Bruto/Realista): {len(X_test)} amostras (Simulações Inéditas).")

# ==============================================================================
# 3. OTIMIZAÇÃO COM OPTUNA
# ==============================================================================
print("\n⚙️ Buscando hiperparâmetros ideais...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'tree_method': 'hist',
        'device': 'cuda' if USAR_CUDA else 'cpu',
        'random_state': 42
    }
    modelo = xgb.XGBClassifier(**params, eval_metric='logloss')

    # Validação Cruzada por Grupo (3-Fold)
    gkf = GroupKFold(n_splits=3)
    score = cross_val_score(modelo, X_train, y_train, cv=gkf, groups=groups_train, scoring='accuracy')
    return score.mean()

estudo = optuna.create_study(direction='maximize')
estudo.optimize(objective, n_trials=10)  # 10 trials para rapidez, mude para 30 para elite

# ==============================================================================
# 4. TREINAMENTO FINAL E ANÁLISE DE THRESHOLD (LIMIAR)
# ==============================================================================
print(f"🧠 Treinando Porteiro Final com Melhores Parâmetros...")
modelo_global = xgb.XGBClassifier(
    **estudo.best_params,
    tree_method='hist',
    device='cuda' if USAR_CUDA else 'cpu',
    random_state=42
)
modelo_global.fit(X_train, y_train)
y_pred = modelo_global.predict(X_test)
acc_final = accuracy_score(y_test, y_pred)

print(f"\n⭐ ACURÁCIA DO PORTEIRO EM DADOS INÉDITOS: {acc_final * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomalia']))

# --- ANÁLISE DE SENSIBILIDADE ---
print("\n🔍 Analisando Limiares de Decisão (Thresholds):")
y_probs = modelo_global.predict_proba(X_test)[:, 1]

resultados_t = []
thresholds = [0.1, 0.2, 0.25, 0.3,0.35, 0.4,0.45, 0.5]

for t in thresholds:
    y_pred_t = (y_probs >= t).astype(int)

    # Taxa de Falso Alarme (FAR): Normal classificada como Anomalia
    far = (y_pred_t[y_test == 0] == 1).mean() * 100

    # Taxa de Detecção de Falhas (FDR): Falha classificada como Anomalia
    fdr = (y_pred_t[y_test == 1] == 1).mean() * 100

    acc_t = accuracy_score(y_test, y_pred_t) * 100
    resultados_t.append({
        'Threshold': t,
        'Detecção (FDR)': fdr,
        'Alarme Falso (FAR)': far,
        'Acurácia': acc_t
    })

df_sensibilidade = pd.DataFrame(resultados_t)
print(df_sensibilidade.to_string(index=False))

# ==============================================================================
# 4.1 SELEÇÃO DE THRESHOLD COM TOPSIS
# ==============================================================================
print("\n📐 Aplicando TOPSIS para selecionar o melhor threshold...")

# Critérios:
# - FDR: benefício
# - Acurácia: benefício
# - FAR: custo
colunas_beneficio = ['Detecção (FDR)', 'Acurácia']
colunas_custo = ['Alarme Falso (FAR)']

# Pesos iguais por padrão
pesos_topsis = [0.55, 0.3, 0.15]

df_sensibilidade['Score TOPSIS'] = aplicar_topsis(
    df_sensibilidade,
    colunas_beneficio=colunas_beneficio,
    colunas_custo=colunas_custo,
    pesos=pesos_topsis
)

df_sensibilidade['Rank TOPSIS'] = df_sensibilidade['Score TOPSIS'].rank(ascending=False, method='dense')
df_sensibilidade = df_sensibilidade.sort_values(by='Score TOPSIS', ascending=False).reset_index(drop=True)

print("\n📊 Resultado TOPSIS por threshold:")
print(df_sensibilidade.to_string(index=False))

# Threshold escolhido pelo melhor score TOPSIS
melhor_t = float(df_sensibilidade.loc[0, 'Threshold'])
print(f"\n🏁 Melhor threshold segundo TOPSIS: {melhor_t}")

# ==============================================================================
# 5. SALVAMENTO ATUALIZADO
# ==============================================================================
pacote_global = {
    'tipo': 'detector_global_elite',
    'modelo': modelo_global,
    'features': features,
    'acuracia_referencia': acc_final,
    'threshold_sugerido': melhor_t,  # Agora definido via TOPSIS
    'tabela_sensibilidade': df_sensibilidade,
    'falhas_excluidas': FALHAS_EXCLUIDAS,
    'metodo_escolha_threshold': 'TOPSIS',
    'criterios_topsis': {
        'beneficio': colunas_beneficio,
        'custo': colunas_custo,
        'pesos': pesos_topsis
    }
}

caminho_save = os.path.join(PASTA_MODELOS, "detector_global_LSH.joblib")
joblib.dump(pacote_global, caminho_save)

# --- CALCULA O TEMPO AQUI NO FINAL ---
tempo_fim = time.time()
tempo_total_minutos = (tempo_fim - tempo_inicio) / 60

print(f"\n💾 DETECTOR GLOBAL salvo com análise de threshold em: {caminho_save}")
print(f"⏱️ TEMPO TOTAL DE EXECUÇÃO: {tempo_total_minutos:.2f} minutos.\n")